## IMPORTS & FUNCTIONS

In [26]:
import os
import glob
from pathlib import Path
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
import pandas as pd

from flopa.io.ptuio.reader import TTTRReader
from flopa.io.ptuio.reconstructor import ScanConfig, ImageReconstructor
from flopa.io.ptuio.utils import estimate_tcspc_bins, aggregate_dataset, average_phasor
from flopa.io.ptuio.decoder import T3OverflowCorrector

def is_mask_useful(mask_path):
    """Returns True if the mask contains at least one labeled pixel (non-zero)."""
    try:
        mask_data = tifffile.imread(str(mask_path))
        # np.any is extremely fast for checking if any non-zero exists
        return np.any(mask_data > 0)
    except Exception:
        return False

def assign_masks(ptu_files, mask_files):
    """
    Assigns each mask to the PTU with the longest matching prefix.
    Solves the B16par_10 vs B16par_10a ambiguity.
    """
    ptu_stems = [p.stem for p in ptu_files]
    # Sort by length descending so we check the most specific (longest) names first
    sorted_stems = sorted(ptu_stems, key=len, reverse=True)
    
    assignment = {stem: [] for stem in ptu_stems}
    
    for m_file in mask_files:
        m_stem = m_file.stem
        for p_stem in sorted_stems:
            if m_stem.startswith(p_stem):
                # Assign to the longest matching PTU stem and stop looking
                assignment[p_stem].append(m_file)
                break
                
    return assignment


def process_full_ptu(file_path, manual_res):
    """Reconstructs the full 1024x1024 dataset once."""
    reader = TTTRReader(str(file_path))
    tags = reader.header.tags
    
    # Resolution logic
    header_res = tags.get("MeasDesc_Resolution", 5e-12)
    tcspc_res = manual_res if manual_res is not None else header_res
    
    # Binning logic (with buffer to prevent overflow warnings)
    sync = tags.get("TTResult_SyncRate", 40e6)
    bins = int(1.0 / (sync * tcspc_res)) + 100 if manual_res else estimate_tcspc_bins(tags) + 100
    
    cfg = ScanConfig(
        lines=LINES, pixels=PIXELS, frames=FRAMES, line_accumulations=LINE_ACCUMULATIONS,
        bidirectional=BIDIRECTIONAL, max_detector=MAX_DETECTOR,
        bidirectional_phase_shift=PHASE_SHIFT if BIDIRECTIONAL else 0
    )
    
    # outputs=None gets all available data (Counts, Lifetime, Phasors G/S)
    reconstructor = ImageReconstructor(config=cfg, tcspc_channels=bins, outputs=None)
    corrector = T3OverflowCorrector(wraparound=tags.get("TTResultFormat_WrapAround", 1024))

    for chunk in reader.iter_chunks():
        reconstructor.update(corrector.correct(chunk))
    
    return reconstructor.finalize(), tcspc_res

## USER CONFIGURATION

In [27]:
# --- FOLDERS ---
PTU_FOLDER = "../test_data/batch_phasor/" 
MASK_FOLDER = "../test_data/batch_phasor/masks/"


# --- REQUIRED SCAN PARAMETERS ---
LINES = 1024
PIXELS = 1024
LINE_ACCUMULATIONS = (16,)
FRAMES = 2

# --- OPTIONAL SCAN PARAMETERS ---
BIDIRECTIONAL = True
PHASE_SHIFT = 0.00666  # Only used if BIDIRECTIONAL is True
MAX_DETECTOR = 2
TCSPC_RESOLUTION = None

# --- DIMENSION SELECTION (Filtering) ---
# Use None to keep everything, or a list of indices to select specific ones
SELECT_DETECTORS = None     # e.g., [0] only process the first detector
SELECT_FRAMES = None        # e.g., [0, 1, 2]
SELECT_SEQUENCES = None     
AGG_CONFIG = (True, True, False) # (Frames, Sequences, Detectors)

# --- CALIBRATION ---
# Enter as complex(real, imag). Use complex(1.0, 0.0) for no change.
CALIBRATION_FACTOR = complex(1.15, 0.0)

## BATCH EXECUTION

In [28]:
ptu_files = list(Path(PTU_FOLDER).glob("*.ptu"))
mask_files = list(Path(MASK_FOLDER).glob("*.tif"))
mask_map = assign_masks(ptu_files, mask_files)

results_list = []
agg_rename_map = {}

for p_file in ptu_files:
    # Match masks (finding all masks that start with the PTU name)
    #matching_masks = [m for m in mask_files if m.stem.startswith(p_file.stem)]
    p_stem = p_file.stem
    matching_masks = mask_map.get(p_stem, [])
    if not matching_masks: continue

    valid_masks = [m for m in matching_masks if is_mask_useful(m)]
    
    if not valid_masks:
        print(f"Skipping {p_file.name}: {len(matching_masks)} mask(s) found, but all are empty.")
        continue

    print(f"--- Processing: {p_file.name} ---")
    
    # 1. Reconstruct Full Image once per PTU
    full_ds, tcspc_res = process_full_ptu(p_file, TCSPC_RESOLUTION)

    # 2. Apply Coordinate Slicing
    if SELECT_DETECTORS is not None: full_ds = full_ds.sel(detector=SELECT_DETECTORS)
    if SELECT_FRAMES is not None: full_ds = full_ds.sel(frame=SELECT_FRAMES)
    if SELECT_SEQUENCES is not None: full_ds = full_ds.sel(sequence=SELECT_SEQUENCES)

    # 3. Apply Aggregation (Summing/Averaging)
    dim_names = ['frame', 'sequence', 'detector']
    for i, is_agg in enumerate(AGG_CONFIG):
        dim = dim_names[i]
        if is_agg:
            n = full_ds.sizes[dim]
            # Map "frame" -> "frame_agg3" if it has 3 elements
            agg_rename_map[dim] = f"{dim}_agg{n}"

    # 3. Aggregation
    dims_to_agg = [dim_names[i] for i, flag in enumerate(AGG_CONFIG) if flag]
    if dims_to_agg:
        full_ds = aggregate_dataset(full_ds, dims_to_agg)

    # 4. Apply each matching mask
    for m_file in matching_masks:
        print(f"  Applying Mask: {m_file.name}")
        mask = tifffile.imread(str(m_file)).astype(np.uint16)
        labels = np.unique(mask)
        labels = labels[labels != 0] # Skip background (0)

        for label_id in labels:
            label_mask = (mask == label_id)
            
            # Iterate through the dimensions left after aggregation
            for f in range(full_ds.sizes['frame']):
                for s in range(full_ds.sizes['sequence']):
                    for d in range(full_ds.sizes['detector']):
                        
                        sub = full_ds.isel(frame=f, sequence=s, detector=d)
                        
                        # Extract 2D arrays
                        cts = sub.photon_count.values
                        lt_img = sub.mean_arrival_time.values

                        # Calculate photon count for this specific label
                        valid_pixels = np.isfinite(lt_img) & (cts > 0) & label_mask
                        label_photons = np.sum(cts[valid_pixels])
                        #label_photons = np.sum(cts[label_mask])
                        if label_photons == 0: continue

                        # CALCULATE PHASOR (G + iS)
                        # We use the raw images to calculate a weighted average for this label
                        ph_img = sub.phasor_g.values + 1j * sub.phasor_s.values
                        avg_ph_raw = average_phasor(ph_img, cts, mask=label_mask)
                        
                        # Apply Calibration
                        avg_ph_cal = avg_ph_raw * CALIBRATION_FACTOR
                        
                        # Calculate Weighted Average Lifetime (ns)
                        avg_tau_bins = np.sum(lt_img[valid_pixels] * cts[valid_pixels]) / label_photons
                        avg_tau_ns = avg_tau_bins * tcspc_res * 1e9

                        results_list.append({
                            "ptu_file": p_file.name,
                            "mask_file": m_file.name,
                            "label": int(label_id),
                            "frame": int(sub.frame),
                            "sequence": int(sub.sequence),
                            "detector": int(sub.detector),
                            "photon_count": int(label_photons),
                            "avg_ns": float(avg_tau_ns),
                            "g_raw": float(avg_ph_raw.real),
                            "s_raw": float(avg_ph_raw.imag),
                            "g_calibrated": float(avg_ph_cal.real),
                            "s_calibrated": float(avg_ph_cal.imag)
                        })

# 5. Save Results to Table
if results_list:
    df = pd.DataFrame(results_list)
    df = df.rename(columns=agg_rename_map)
    df.to_csv(f'{PTU_FOLDER}/measurements.csv', index=False)
    print(f"\nSuccess! Table saved to: '{PTU_FOLDER}/measurements.csv'")
else:
    print("No data was processed. Check your mask names or photon counts.")

--- Processing: B16par_10.ptu ---
TCSPC channel overflow detected! Max channel: 24147
Channel overflow detected! Max channel: 50
  Applying Mask: B16par_10_d1.tif
  Applying Mask: B16par_10_m1.tif
--- Processing: B16par_10a.ptu ---
TCSPC channel overflow detected! Max channel: 24147
Channel overflow detected! Max channel: 50
  Applying Mask: B16par_10ai1.tif
  Applying Mask: B16par_10a_empty.tif

Success! Table saved to: '../test_data/batch_phasor//measurements.csv'
